In [1]:
import numpy as np
import pandas as pd

from SymbolicDSGE import DSGESolver, ModelParser, Shock
from SymbolicDSGE.monte_carlo import (
    MCPipeline,
    reference_filter_step,
    simulation_step,
    ljung_box_test_step,
    wald_test_step,
    regression_step,
)
import cProfile

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
model, kalman = ModelParser("../../MODELS/misspec_test/reference.yaml").get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(
    n_state=3,
    n_exog=3,
)

steady_state = np.zeros(5, dtype=np.float64)
reference = solver.solve(compiled, steady_state=steady_state)

dgp_model, dgp_kalman = ModelParser("../../MODELS/misspec_test/misspec.yaml").get_all()
dgp_comp = DSGESolver(dgp_model, dgp_kalman).compile(n_state=3, n_exog=3)
dgp_sol = DSGESolver(dgp_model, dgp_kalman)
dgp = dgp_sol.solve(dgp_comp, steady_state=steady_state)

In [3]:
T = 1000
n_obs = len(reference.compiled.observable_names)

pipeline = MCPipeline(
    [
        simulation_step(
            T=T,
            shocks={
                "g,z": Shock(T=T, dist="norm", multivar=True, seed=0),
                "r": Shock(T=T, dist="norm", seed=1),
            },
            observables=True,
        ),
        reference_filter_step(estimate_R_diag=False),
        wald_test_step(
            "std_innov_mean",
            source="std_innov",
            target=np.zeros(n_obs),
            kind="mean",
            burn_in=20,
        ),
        ljung_box_test_step(
            "OGap_autocorr",
            source="std_innov",
            column=0,
            lags=10,
            burn_in=20,
        ),
        ljung_box_test_step(
            "Infl_autocorr",
            source="std_innov",
            column=1,
            lags=10,
            burn_in=20,
        ),
        ljung_box_test_step(
            "Rate_autocorr",
            source="std_innov",
            column=2,
            lags=10,
            burn_in=20,
        ),
        wald_test_step(
            "std_innov_second_moment",
            source="std_innov",
            target=np.eye(n_obs),
            kind="second_moment",
            burn_in=20,
        ),
        regression_step(
            "OutGap_regression",
            kind="ridge_gs",
            start=1e-6,
            stop=1e2,
            num=10,
            X_source="x_pred",
            y_source="innov",
            y_column=0,
            X_columns=[2, 3, 4],
            variables=["r", "x", "Pi"],
        ),
    ]
)

In [8]:
run = lambda: pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=1000,
    retain_payloads=False,
    retain_test_results=False,
    verbosity=2,
)
cProfile.run("mc = run()", sort="time")
mc.succeeded, mc.n_successful

datagen concluded successfully with 232.48 it/s.
filter concluded successfully with 447.87 it/s.
std_innov_mean concluded successfully with 4334.37 it/s.
OGap_autocorr concluded successfully with 19244.79 it/s.
Infl_autocorr concluded successfully with 24392.92 it/s.
Rate_autocorr concluded successfully with 26248.51 it/s.
std_innov_second_moment concluded successfully with 1825.04 it/s.
OutGap_regression concluded successfully with 2067.35 it/s.
         1812630 function calls (1808614 primitive calls) in 8.027 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1000    3.149    0.003    4.238    0.004 solved_model.py:55(sim)
     1000    1.572    0.002    1.624    0.002 filter.py:942(run)
     2000    0.377    0.000    0.505    0.000 hac_covariance.py:270(hac_covariance)
     1000    0.355    0.000    0.425    0.000 core.py:102(ridge_gs)
     2000    0.175    0.000    0.812    0.000 shock_generators.py:15(abstract_shock_

(True, 1000)

In [5]:
t_summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.test_summaries.items()
    }
).T
t_summary.round(3)

,mean_statistic,mean_pval,rejection_rate,ci_low,ci_high
std_innov_mean,2.531,0.569,0.035,0.025,0.048
OGap_autocorr,88.115,0.000,1.000,0.996,1.000
Infl_autocorr,125.037,0.000,1.000,0.996,1.000
Rate_autocorr,22.446,0.076,0.673,0.643,0.701
std_innov_second_moment,3000.276,0.000,1.000,0.996,1.000


In [6]:
print(
    any(
        res.value != 0
        for res in mc.regression_summaries["OutGap_regression"].status_trace
    )
)

(
    mc.regression_summaries["OutGap_regression"].coefficients.mean(axis=0).round(2),
    mc.regression_summaries["OutGap_regression"].r2_trace.mean().round(4),
)

False


(array([ 0.  , -0.19, -0.05, -3.79]), np.float64(0.1186))